In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
workdir = '../../../hybrid_diversity/4_read_mapping/3_deduplicated/2_nuc_and_chloroplast/'
infile = workdir + '12B1_10kb.all_samples_counts.tab'    

# define references
reference_samples = ["CC2192B5", "CC2192D3"]

# #outfiles
# # Per-strain log2ratio files will be written as: workdir + '{strain}.log2ratio.tsv'
# medfile = workdir + '12B1_10kb.chr_median_log2_ratios.tsv'

In [3]:
chromdict = {'CM083856.1': 'scaf1',
'CM083857.1': 'scaf2',
'CM083858.1': 'scaf3',
'CM083859.1': 'scaf4',
'CM083860.1': 'scaf5',
'CM083861.1': 'scaf6',
'CM083862.1': 'scaf7',
'CM083863.1': 'scaf8',
'CM083864.1': 'scaf9',
'CM083865.1': 'scaf10',
'CM083866.1': 'scaf11',
'CM083867.1': 'scaf12',
'CM083868.1': 'scaf13',
'CM083869.1': 'scaf14',
'CM083870.1': 'scaf15',
'CM083871.1': 'scaf16',
'CM083872.1': 'scaf17',
'CM083873.1': 'scaf18',
'CM083874.1': 'scaf19',
'CM083875.1': 'scaf20',
'CM083876.1': 'scaf21',
'CM083877.1': 'scaf22',
'CM083878.1': 'scaf23',
'CM083879.1': 'scaf24',
'CM083880.1': 'scaf25',
'CM083881.1': 'scaf26',
'CM083882.1': 'scaf27',
'CM083883.1': 'scaf28',
'CM083884.1': 'scaf29',
'CM083885.1': 'scaf30',
'CM083886.1': 'scaf31',
'CM083887.1': 'scaf32',
'CM083888.1': 'scaf33',
'CM083889.1': 'scaf34'}

In [4]:
# Load raw counts
counts = pd.read_csv(infile, sep="\t")
counts.columns = counts.columns.str.strip("'#").str.strip()

# Remove chloroplast scaffold
counts = counts[counts["chr"] != "scf7180000001165"]

# Remove small scaffolds 31-34
counts = counts[counts["chr"] != "CM083886.1"]
counts = counts[counts["chr"] != "CM083887.1"]
counts = counts[counts["chr"] != "CM083888.1"]
counts = counts[counts["chr"] != "CM083889.1"]
counts = counts[counts["chr"] != "CM083885.1"]

# Rename chromosomes 
counts["chr"] = counts["chr"].replace(chromdict)
all_samples = counts.columns[3:].tolist()
#counts

all_samples = counts.columns[3:].tolist()

# Natural chromosome sort
chr_summary = (
    counts
    .groupby("chr")[all_samples]
    .median()
)

chr_summary = chr_summary.loc[
    sorted(chr_summary.index,
           key=lambda x: (0, int(x.replace("scaf","").split("_")[0]))
                          if x.replace("scaf","").split("_")[0].isdigit()
                          else (1, x))
]
#chr_summary

In [5]:
# # Within-sample normalization 
# sample_data = counts[all_samples]
# within_normalized = sample_data.div(sample_data.median(axis=0), axis=1)

# norm_df = counts[["chr", "start", "end"]].copy()
# norm_df = pd.concat([norm_df, within_normalized], axis=1)

In [6]:
# Diploid mean from references only
diploid_mean = counts[reference_samples].mean(axis=1)
diploid_mean

0       73.217651
1       92.662098
2       69.585651
3       80.201298
4       74.658398
          ...    
8893    83.054501
8894    81.397049
8895    80.788250
8896    80.691898
8897     7.371020
Length: 8898, dtype: float64

In [8]:
# ── Log2 ratios for all non-reference samples ─────────────────────────────────
suspect_samples = [s for s in all_samples if s not in reference_samples]

pseudo = 0.01

for sample in suspect_samples:
    sample_log2 = counts[["chr", "start", "end"]].copy()
    sample_log2["log2ratio"] = np.log2(
        (counts[sample] + pseudo) / (diploid_mean + pseudo)
    )
    # Sanitize sample name for use in filename (replace / and spaces)
    safe_name = sample.replace("/", "_").replace(" ", "_")
    outfile = workdir + f'{safe_name}.log2ratio.tsv'
    print(outfile)
    sample_log2.to_csv(outfile, sep="\t", index=False)

print(f"Wrote {len(suspect_samples)} files to {workdir}")
print("Example output (first sample):")
sample_log2

../../../hybrid_diversity/4_read_mapping/3_deduplicated/2_nuc_and_chloroplast/12A1.log2ratio.tsv
../../../hybrid_diversity/4_read_mapping/3_deduplicated/2_nuc_and_chloroplast/12A1-SC1.log2ratio.tsv
../../../hybrid_diversity/4_read_mapping/3_deduplicated/2_nuc_and_chloroplast/12A1-SC2.log2ratio.tsv
../../../hybrid_diversity/4_read_mapping/3_deduplicated/2_nuc_and_chloroplast/12A1-SC3.log2ratio.tsv
../../../hybrid_diversity/4_read_mapping/3_deduplicated/2_nuc_and_chloroplast/12B1.log2ratio.tsv
../../../hybrid_diversity/4_read_mapping/3_deduplicated/2_nuc_and_chloroplast/13A5.log2ratio.tsv
../../../hybrid_diversity/4_read_mapping/3_deduplicated/2_nuc_and_chloroplast/8020.log2ratio.tsv
../../../hybrid_diversity/4_read_mapping/3_deduplicated/2_nuc_and_chloroplast/8027.log2ratio.tsv
../../../hybrid_diversity/4_read_mapping/3_deduplicated/2_nuc_and_chloroplast/ARC66.log2ratio.tsv
../../../hybrid_diversity/4_read_mapping/3_deduplicated/2_nuc_and_chloroplast/ARC81.log2ratio.tsv
../../../hybrid_

,chr,start,end,log2ratio
0,scaf1,0,10000,0.485051
1,scaf1,10000,20000,0.353815
2,scaf1,20000,30000,0.339083
3,scaf1,30000,40000,0.307895
4,scaf1,40000,50000,-0.345528
...,...,...,...,...
8893,scaf29,1230000,1240000,0.264516
8894,scaf29,1240000,1250000,-0.347440
8895,scaf29,1250000,1260000,-0.212276
8896,scaf29,1260000,1270000,0.043702
